# Full Model Training Pipeline

This notebook trains **9 XGBoost models** - one for each combination of:
- **Tag Groups**: Genre, Mood, Situation
- **Optimization Metrics**: Precision, Recall, F1

## Pipeline Steps
1. Load training dataset
2. Hyperparameter tuning (100 random trials per model)
3. Threshold optimization per metric
4. Performance comparison across tag groups
5. Save all trained models

In [ ]:
# Setup and imports
%load_ext autoreload
%autoreload 2

from pyrekordbox import Rekordbox6Database
import polars as pl
import numpy as np
from nbutils import setup_path, display_polars

setup_path()

from utils import get_base_dataset
from processing import preprocess_tag_group
from models import (
    tune_xgboost_hyperparams,
    optimize_prediction_thresholds,
    save_model,
)

db = Rekordbox6Database()
pl.Config.set_tbl_rows(30)

## 1. Data Loading and Preparation

Load the base dataset and audio features, then prepare train/validation/test splits.

In [ ]:
# Get base dataset
base_df = get_base_dataset(db, min_tag_count=5)

# Load audio features
features_df = pl.read_parquet("../data/song_features.parquet")

# Define feature columns to exclude
exclude_cols = [
    "song_path",
    "harmonic_percussive_ratio",
    "percussive_strength",
    "tonnetz_mean_0", "tonnetz_mean_1", "tonnetz_mean_2", "tonnetz_mean_3", "tonnetz_mean_4", "tonnetz_mean_5",
    "tonnetz_std_0", "tonnetz_std_1", "tonnetz_std_2", "tonnetz_std_3", "tonnetz_std_4", "tonnetz_std_5"
]

feature_cols = [col for col in features_df.columns 
                if col not in exclude_cols + ["song_id", "song_path"]]

# Filter features to remove null values
features_df = features_df.filter(pl.col("energy_increase_ratio").is_not_null())

print(f"\nFeature matrix shape: {features_df.shape}")
print(f"Number of features: {len(feature_cols)}")

## 2. Train Models for All Tag Groups and Metrics

We'll train 9 models total:
- Genre: precision, recall, f1
- Mood: precision, recall, f1  
- Situation: precision, recall, f1

In [ ]:
# Configuration
TAG_GROUPS = ["Genre", "Mood", "Situation"]
METRICS = ["precision", "recall", "f1"]
N_TRIALS = 100  # Random search iterations per model

# Store all results
all_models = {}
all_threshold_results = {}
all_preprocessed_data = {}

print(f"Training {len(TAG_GROUPS)} × {len(METRICS)} = {len(TAG_GROUPS) * len(METRICS)} models")
print(f"Each model: {N_TRIALS} hyperparameter trials + threshold optimization")

### 2.1 Genre Models

In [ ]:
# Preprocess Genre data
print("="*80)
print("PREPROCESSING: GENRE")
print("="*80)

genre_result = preprocess_tag_group(
    base_df, features_df, "Genre",
    feature_cols=feature_cols,
    test_size=0.2,
    val_size=0.2,
    min_train_count=10,
    apply_scaling=True,
    apply_pca=False,
)

all_preprocessed_data["Genre"] = genre_result

print(f"\nGenre - Data split:")
print(f"  Train: {genre_result.X_train.shape[0]} samples")
print(f"  Validation: {genre_result.X_val.shape[0]} samples")
print(f"  Test: {genre_result.X_test.shape[0]} samples")
print(f"  Labels: {len(genre_result.tags)}")

In [ ]:
# Genre - Precision Model
print("\n" + "="*80)
print("GENRE - PRECISION MODEL")
print("="*80)

# Hyperparameter tuning optimized for precision
genre_precision_results = tune_xgboost_hyperparams(
    X_train=genre_result.X_train,
    y_train=genre_result.y_train,
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='precision',
    verbose=True
)

# Threshold optimization for precision
genre_precision_thresholds = optimize_prediction_thresholds(
    model=genre_precision_results['best_model'],
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    metric='precision',
    verbose=True
)

# Store results
all_models['genre_precision'] = genre_precision_results
all_threshold_results['genre_precision'] = genre_precision_thresholds

In [ ]:
# Genre - Recall Model
print("\n" + "="*80)
print("GENRE - RECALL MODEL")
print("="*80)

# Hyperparameter tuning optimized for recall
genre_recall_results = tune_xgboost_hyperparams(
    X_train=genre_result.X_train,
    y_train=genre_result.y_train,
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='recall',
    verbose=True
)

# Threshold optimization for recall
genre_recall_thresholds = optimize_prediction_thresholds(
    model=genre_recall_results['best_model'],
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    metric='recall',
    verbose=True
)

# Store results
all_models['genre_recall'] = genre_recall_results
all_threshold_results['genre_recall'] = genre_recall_thresholds

In [ ]:
# Genre - F1 Model
print("\n" + "="*80)
print("GENRE - F1 MODEL")
print("="*80)

# Hyperparameter tuning optimized for F1
genre_f1_results = tune_xgboost_hyperparams(
    X_train=genre_result.X_train,
    y_train=genre_result.y_train,
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='f1',
    verbose=True
)

# Threshold optimization for f1
genre_f1_thresholds = optimize_prediction_thresholds(
    model=genre_f1_results['best_model'],
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    metric='f1',
    verbose=True
)

# Store results
all_models['genre_f1'] = genre_f1_results
all_threshold_results['genre_f1'] = genre_f1_thresholds

### 2.2 Mood Models

In [ ]:
# Preprocess Mood data
print("="*80)
print("PREPROCESSING: MOOD")
print("="*80)

mood_result = preprocess_tag_group(
    base_df, features_df, "Mood",
    feature_cols=feature_cols,
    test_size=0.2,
    val_size=0.2,
    min_train_count=10,
    apply_scaling=True,
    apply_pca=False,
)

all_preprocessed_data["Mood"] = mood_result

print(f"\nMood - Data split:")
print(f"  Train: {mood_result.X_train.shape[0]} samples")
print(f"  Validation: {mood_result.X_val.shape[0]} samples")
print(f"  Test: {mood_result.X_test.shape[0]} samples")
print(f"  Labels: {len(mood_result.tags)}")

In [ ]:
# Mood - Precision Model
print("\n" + "="*80)
print("MOOD - PRECISION MODEL")
print("="*80)

mood_precision_results = tune_xgboost_hyperparams(
    X_train=mood_result.X_train,
    y_train=mood_result.y_train,
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='precision',
    verbose=True
)

mood_precision_thresholds = optimize_prediction_thresholds(
    model=mood_precision_results['best_model'],
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    metric='precision',
    verbose=True
)

all_models['mood_precision'] = mood_precision_results
all_threshold_results['mood_precision'] = mood_precision_thresholds

In [ ]:
# Mood - Recall Model
print("\n" + "="*80)
print("MOOD - RECALL MODEL")
print("="*80)

mood_recall_results = tune_xgboost_hyperparams(
    X_train=mood_result.X_train,
    y_train=mood_result.y_train,
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='recall',
    verbose=True
)

mood_recall_thresholds = optimize_prediction_thresholds(
    model=mood_recall_results['best_model'],
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    metric='recall',
    verbose=True
)

all_models['mood_recall'] = mood_recall_results
all_threshold_results['mood_recall'] = mood_recall_thresholds

In [ ]:
# Mood - F1 Model
print("\n" + "="*80)
print("MOOD - F1 MODEL")
print("="*80)

mood_f1_results = tune_xgboost_hyperparams(
    X_train=mood_result.X_train,
    y_train=mood_result.y_train,
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='f1',
    verbose=True
)

mood_f1_thresholds = optimize_prediction_thresholds(
    model=mood_f1_results['best_model'],
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    metric='f1',
    verbose=True
)

all_models['mood_f1'] = mood_f1_results
all_threshold_results['mood_f1'] = mood_f1_thresholds

### 2.3 Situation Models

In [ ]:
# Preprocess Situation data
print("="*80)
print("PREPROCESSING: SITUATION")
print("="*80)

situation_result = preprocess_tag_group(
    base_df, features_df, "Situation",
    feature_cols=feature_cols,
    test_size=0.2,
    val_size=0.2,
    min_train_count=10,
    apply_scaling=True,
    apply_pca=False,
)

all_preprocessed_data["Situation"] = situation_result

print(f"\nSituation - Data split:")
print(f"  Train: {situation_result.X_train.shape[0]} samples")
print(f"  Validation: {situation_result.X_val.shape[0]} samples")
print(f"  Test: {situation_result.X_test.shape[0]} samples")
print(f"  Labels: {len(situation_result.tags)}")

In [ ]:
# Situation - Precision Model
print("\n" + "="*80)
print("SITUATION - PRECISION MODEL")
print("="*80)

situation_precision_results = tune_xgboost_hyperparams(
    X_train=situation_result.X_train,
    y_train=situation_result.y_train,
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='precision',
    verbose=True
)

situation_precision_thresholds = optimize_prediction_thresholds(
    model=situation_precision_results['best_model'],
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    metric='precision',
    verbose=True
)

all_models['situation_precision'] = situation_precision_results
all_threshold_results['situation_precision'] = situation_precision_thresholds

In [ ]:
# Situation - Recall Model
print("\n" + "="*80)
print("SITUATION - RECALL MODEL")
print("="*80)

situation_recall_results = tune_xgboost_hyperparams(
    X_train=situation_result.X_train,
    y_train=situation_result.y_train,
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='recall',
    verbose=True
)

situation_recall_thresholds = optimize_prediction_thresholds(
    model=situation_recall_results['best_model'],
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    metric='recall',
    verbose=True
)

all_models['situation_recall'] = situation_recall_results
all_threshold_results['situation_recall'] = situation_recall_thresholds

In [ ]:
# Situation - F1 Model
print("\n" + "="*80)
print("SITUATION - F1 MODEL")
print("="*80)

situation_f1_results = tune_xgboost_hyperparams(
    X_train=situation_result.X_train,
    y_train=situation_result.y_train,
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='f1',
    verbose=True
)

situation_f1_thresholds = optimize_prediction_thresholds(
    model=situation_f1_results['best_model'],
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    metric='f1',
    verbose=True
)

all_models['situation_f1'] = situation_f1_results
all_threshold_results['situation_f1'] = situation_f1_thresholds

## 3. Performance Comparison

Compare all models across tag groups and optimization metrics.

In [ ]:
# Create comprehensive comparison table
comparison_data = []

for model_key, threshold_result in all_threshold_results.items():
    tag_group, metric = model_key.rsplit('_', 1)
    
    # Get preprocessed data for this tag group
    tag_group_name = tag_group.capitalize()
    data = all_preprocessed_data[tag_group_name]
    
    comparison_data.append({
        "tag_group": tag_group_name,
        "optimization_metric": metric,
        "model_name": f"xgboost_{tag_group}_{metric}",
        "n_tags": len(data.tags),
        "tags": ", ".join(data.tags),
        "train_support": data.X_train.shape[0],
        "val_support": data.X_val.shape[0],
        "test_support": data.X_test.shape[0],
        "macro_precision": threshold_result['optimized_metrics']['macro_precision'],
        "macro_recall": threshold_result['optimized_metrics']['macro_recall'],
        "macro_f1": threshold_result['optimized_metrics']['macro_f1'],
        "weighted_precision": threshold_result['optimized_metrics']['weighted_precision'],
        "weighted_recall": threshold_result['optimized_metrics']['weighted_recall'],
        "weighted_f1": threshold_result['optimized_metrics']['weighted_f1'],
        "default_f1": threshold_result['default_metrics']['macro_f1'],
        "f1_improvement": threshold_result['improvement'],
    })

comparison_df = pl.DataFrame(comparison_data)

print("\n" + "="*100)
print("MODEL PERFORMANCE COMPARISON")
print("="*100)
print("\nAll 9 Models - Sorted by Tag Group and Optimization Metric:")
display_polars(comparison_df.sort(["tag_group", "optimization_metric"]), lim=20)

In [ ]:
# Compare by optimization metric
print("\n" + "="*100)
print("PERFORMANCE BY OPTIMIZATION METRIC")
print("="*100)

for metric in METRICS:
    print(f"\n{metric.upper()} Models:")
    metric_models = comparison_df.filter(pl.col("optimization_metric") == metric)
    display_polars(
        metric_models.select([
            "tag_group", "n_tags", 
            "macro_precision", "macro_recall", "macro_f1",
            "weighted_precision", "weighted_recall", "weighted_f1",
            "f1_improvement"
        ]).sort("macro_f1", descending=True),
        lim=10
    )

In [ ]:
# Compare by tag group
print("\n" + "="*100)
print("PERFORMANCE BY TAG GROUP")
print("="*100)

for tag_group in TAG_GROUPS:
    print(f"\n{tag_group.upper()} Models:")
    group_models = comparison_df.filter(pl.col("tag_group") == tag_group)
    display_polars(
        group_models.select([
            "optimization_metric", "n_tags",
            "macro_precision", "macro_recall", "macro_f1",
            "weighted_precision", "weighted_recall", "weighted_f1",
            "f1_improvement"
        ]).sort("optimization_metric"),
        lim=10
    )

In [ ]:
# Show best model per tag group
print("\n" + "="*100)
print("BEST MODEL PER TAG GROUP (by Macro F1 Score)")
print("="*100)

best_models = (
    comparison_df
    .sort("macro_f1", descending=True)
    .group_by("tag_group")
    .first()
    .sort("tag_group")
)

display_polars(
    best_models.select([
        "tag_group", "optimization_metric", "n_tags",
        "macro_precision", "macro_recall", "macro_f1",
        "weighted_precision", "weighted_recall", "weighted_f1"
    ]),
    lim=10
)

## 4. Save All Models

Save all 9 trained models with their configurations, thresholds, and preprocessing components.

In [ ]:
# Save all models
print("\n" + "="*100)
print("SAVING ALL MODELS")
print("="*100)

saved_paths = {}

for model_key in all_models.keys():
    tag_group, metric = model_key.rsplit('_', 1)
    tag_group_name = tag_group.capitalize()
    
    # Get data
    data = all_preprocessed_data[tag_group_name]
    model_results = all_models[model_key]
    threshold_results = all_threshold_results[model_key]
    
    # Save model with both macro and weighted metrics
    paths = save_model(
        model=model_results['best_model'],
        model_name=f"xgboost_{tag_group_name}_{metric}",
        save_dir="../models",
        tags=data.tags,
        thresholds=threshold_results['thresholds'],
        hyperparams=model_results['best_params'],
        scaler=data.scaler,
        pca=data.pca,
        metrics={
            "macro_precision": threshold_results['optimized_metrics']['macro_precision'],
            "macro_recall": threshold_results['optimized_metrics']['macro_recall'],
            "macro_f1": threshold_results['optimized_metrics']['macro_f1'],
            "weighted_precision": threshold_results['optimized_metrics']['weighted_precision'],
            "weighted_recall": threshold_results['optimized_metrics']['weighted_recall'],
            "weighted_f1": threshold_results['optimized_metrics']['weighted_f1'],
            "optimization_metric": metric,
        },
        tag_group=tag_group_name,
        verbose=True
    )
    
    saved_paths[model_key] = paths

print("\n" + "="*100)
print(f"SUCCESSFULLY SAVED {len(saved_paths)} MODELS")
print("="*100)

In [ ]:
# Summary of saved models
print("\nSaved Model Files:")
for model_key, paths in saved_paths.items():
    print(f"\n{model_key}:")
    for file_type, path in paths.items():
        if path:
            print(f"  {file_type}: {path}")

## 5. Training Summary

Final summary of all trained models and their performance.

In [ ]:
print("\n" + "="*100)
print("TRAINING COMPLETE - FINAL SUMMARY")
print("="*100)

print(f"\nTotal models trained: {len(all_models)}")
print(f"Tag groups: {', '.join(TAG_GROUPS)}")
print(f"Optimization metrics: {', '.join(METRICS)}")
print(f"Hyperparameter trials per model: {N_TRIALS}")

print("\nPerformance Summary (Macro Averages):")
summary_stats = comparison_df.select([
    pl.col("macro_f1").mean().alias("avg_macro_f1"),
    pl.col("macro_f1").max().alias("best_macro_f1"),
    pl.col("macro_precision").mean().alias("avg_macro_precision"),
    pl.col("macro_recall").mean().alias("avg_macro_recall"),
    pl.col("weighted_f1").mean().alias("avg_weighted_f1"),
    pl.col("weighted_precision").mean().alias("avg_weighted_precision"),
    pl.col("weighted_recall").mean().alias("avg_weighted_recall"),
    pl.col("f1_improvement").mean().alias("avg_f1_improvement"),
])

print(summary_stats)

print("\nBest Overall Model (by Macro F1):")
best_overall = comparison_df.sort("macro_f1", descending=True).head(1)
display_polars(
    best_overall.select([
        "model_name", "tag_group", "optimization_metric",
        "macro_precision", "macro_recall", "macro_f1",
        "weighted_precision", "weighted_recall", "weighted_f1"
    ]),
    lim=5
)

print("\n" + "="*100)
print("All models saved to: ../models/")
print("="*100)